In [ ]:
import os
os.chdir('/home/linhang/workbench/workbench/Earthquake_predictor/dataset')
os.chdir('/home/linhang/workbench/workbench/Earthquake_predictor/')
from dataset.dataset_utils import *
from model.ES_net_mixer import *
from model import LightingModel
from torch.utils.data import DataLoader
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [ ]:
import os
data_path = "/home/linhang/workbench/Earthquake_data/"
os.listdir(data_path)

In [ ]:
area = 'California (Southern)'
data_area_path = data_path + area + "/"
gnss_data = pd.read_csv(data_area_path + "gnss_data.csv", index_col=0, parse_dates=True, low_memory=False).map(parse_str_list)
earthquake_data = pd.read_csv(data_area_path+"earthquake_data.csv", index_col=0, parse_dates=True)
energy_data = pd.read_csv(data_area_path+"energy_data.csv", index_col=0, parse_dates=True)
station_dict_use = pickle.load(open(data_area_path+"station_dict_use.pkl", "rb"))
earthquake_dict_use = pickle.load(open(data_area_path+"grid_data/grid_id_map.pkl", "rb"))
es_geo_matrix = pd.read_csv(data_area_path+"es_geo_matrix.csv", index_col=0)
es_sem_matrix = pd.read_csv(data_area_path+"es_sem_matrix.csv", index_col=0)
gnss_geo_matrix = pd.read_csv(data_area_path+"gnss_geo_matrix.csv", index_col=0)

In [ ]:
window_size = 140
forecast_horizon = 700
lape_dim = 30
geo_percentage = 0.3
sem_percentage = 0.3
earthquake_catalog_window = 1400
dataset = EarthquakeGNSSDataset(area=area,earthquake_data=earthquake_data,es_geo_matrix=es_geo_matrix,es_sem_matrix=es_sem_matrix,
                                gnss_geo_matrix=gnss_geo_matrix,gnss_data=gnss_data,geo_percentage=geo_percentage, sem_percentage=sem_percentage,
                                lape_dim=lape_dim,earthquake_dict_use = earthquake_dict_use,station_dict_use = station_dict_use,
                                window_size=window_size,forecast_horizon=forecast_horizon,earthquake_threshold=4,missing_threshold=5,earthquake_catalog_window=earthquake_catalog_window)

In [ ]:
def find_activate_area(earthquake_history,topk,earthquake_future):
    energy_catalog_sum = earthquake_future.sum(axis = 1).cpu().numpy()
    topk_index = np.argsort(energy_catalog_sum)[-topk:]
    earthquake_history_topk = earthquake_history[:,topk_index,:].squeeze(-1).cpu().numpy()
    earthquake_future_topk = earthquake_future[topk_index].permute(1,0).cpu().numpy()
    return earthquake_history_topk,earthquake_future_topk,topk_index

In [ ]:
data_history = dataset[100]["log_energy_history"]
data_future = dataset[100]["log_energy_future"]

In [ ]:
earthquake_history_topk,earthquake_future_topk, topk_index = find_activate_area(data_history,4,data_future)

In [ ]:
with open("model_params.json", 'r') as f:
    model_par = json.load(f)
    

In [ ]:
model = model = LightingModel(
                    ES_net_mixer,
                    **model_par)

In [ ]:
for i in range(4):
    plt.plot(range(len(earthquake_history_topk[:,i])),earthquake_history_topk[:,i])
    plt.plot(range(len(earthquake_history_topk[:,i]),len(earthquake_history_topk[:,i])+len(earthquake_future_topk[:,i])),earthquake_future_topk[:,i])
    plt.show()